# 03 — Conventional Machine Learning for Apparent Viscosity

## Machine Learning Formulations
Conventional ML models (Random Forest and XGBoost) predict $\eta_a$ directly from input features:
$$\hat{\eta}_a = f(\dot{\gamma}, c, T, \text{fluid\_type})$$

### Methodological Requirements
1. **Data Leakage Prevention**: Using `GroupShuffleSplit` on experimental condition curves so that all shear-rate points of a single flow curve reside entirely in train or test.
2. **Physical Feasibility Checks**: Enforce non-negative viscosity $\hat{\eta}_a > 0$.
3. **Generalization Evaluation**: Out-of-condition testing (leave-one-concentration-out).
4. **Interpretability**: SHAP (SHapley Additive exPlanations) for global feature importance and interaction dynamics.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path('.').resolve().parent if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import PROCESSED_DATA_DIR, TABLES_DIR
from src.data_processing import load_dataset
from src.ml_models import (
    prepare_features, split_data, create_group_labels,
    train_random_forest, train_xgboost, evaluate_model,
    generalization_experiment
)
from src.explainability import compute_shap_values, plot_shap_summary, plot_shap_bar
from src.visualization import plot_actual_vs_predicted, plot_residuals

df = load_dataset(PROCESSED_DATA_DIR / 'processed_data.csv')
print(f'Loaded {len(df)} records.')

## 1. Grouped Splitting & Feature Encoding

In [ ]:
X, y, num_features, cat_features = prepare_features(df)
groups = create_group_labels(df)
X_train, X_test, y_train, y_test = split_data(X, y, groups=groups, test_size=0.25, random_state=42)
print(f'Train samples: {len(X_train)}, Test samples: {len(X_test)}')
print(f'Numeric features: {num_features}, Categorical: {cat_features}')

## 2. Model Training: Random Forest & XGBoost

In [ ]:
rf_pipe, rf_train_met = train_random_forest(X_train, y_train, num_features, cat_features)
y_pred_rf, rf_test_met = evaluate_model(rf_pipe, X_test, y_test, 'Random Forest')

xgb_pipe, xgb_train_met = train_xgboost(X_train, y_train, num_features, cat_features)
y_pred_xgb, xgb_test_met = evaluate_model(xgb_pipe, X_test, y_test, 'XGBoost')

results_df = pd.DataFrame([
    {'Model': 'Random Forest', **rf_test_met},
    {'Model': 'XGBoost', **xgb_test_met}
])
display(results_df)

## 3. Actual vs Predicted & Residual Diagnostics

In [ ]:
fig_rf_act = plot_actual_vs_predicted(y_test.values, y_pred_rf, 'Random Forest')
plt.show()

fig_rf_res = plot_residuals(y_test.values, y_pred_rf, 'Random Forest')
plt.show()

## 4. Generalization Experiment: Held-Out Concentrations

In [ ]:
gen_file = TABLES_DIR / 'generalization_results.csv'
if gen_file.exists():
    gen_table = pd.read_csv(gen_file)
    display(gen_table)
else:
    gen_results = generalization_experiment(df)
    display(pd.DataFrame(gen_results))

## 5. SHAP Explainability Analysis
Understanding how tree models utilize shear rate, concentration, and temperature.

In [ ]:
rf_expl = compute_shap_values(rf_pipe, X_train, X_test, model_name='Random Forest')
if rf_expl is not None:
    fig_bar = plot_shap_bar(rf_expl, 'Random Forest')
    plt.show()
    fig_sum = plot_shap_summary(rf_expl, 'Random Forest')
    plt.show()

### Discussion: SHAP Insights vs. Causality
- **Feature Dominance**: Shear rate ($\dot{\gamma}$) exhibits the highest global SHAP importance, driving large negative SHAP values as shear rate increases (capturing the pseudoplastic power law trend).
- **Concentration**: Higher concentration exerts positive SHAP values, shifting viscosity upward across all shear rates.
- **Scientific Caveat**: SHAP scores reflect the sensitivity of the tree ensemble's partition thresholds, not mechanistic causality.